# Customer identity resolution: walkthrough

One customer, many IDs. A person buys on the web store under two accounts, then on a marketplace that shares no customer ID and rarely an email. Every channel reports them as a different "customer", so repeat buyers get counted as new, retention looks weaker than it is, and acquisition gets credit it didn't earn.

This notebook runs the pipeline on a **synthetic dataset** (`synthetic/generate_data.py`) that reproduces the messy patterns of real commerce exports. The flow:

1. **Profile the identifiers**: which keys exist per channel, which are junk, and which are trustworthy
2. **Resolve**: apply the match rules and connect the matches into unified IDs
3. **Validate**: check cluster sizes and look for over-merging
4. **Measure the impact**: customers, repeat rate and "new" customers under channel-native vs. unified IDs

The principle: **profile before you design rules.** Every rule in `identity_resolution/resolve.py` traces back to a check in section 1.

## 0. Setup

In [1]:
import sys, os, duckdb, pandas as pd
sys.path.insert(0, "..")
from identity_resolution.resolve import run, RULES, MAX_ENTITIES
pd.set_option("display.max_columns", None, "display.width", 200)

DATA = "../data/*.csv.gz"   # python synthetic/generate_data.py --out data
con = duckdb.connect()
q = lambda s: con.sql(s).df()
con.sql(f"""CREATE TABLE raw AS SELECT * FROM read_csv('{DATA}', header=true, all_varchar=true, union_by_name=true)""")
q("SELECT lower(channel) AS channel, count(*) AS orders FROM raw GROUP BY 1")

,channel,orders
0,amazon,6902
1,shopify,27149


## 1. Profile the identifiers
### 1a. Coverage by channel
The keys available on **both** sides are the only ones that can link the channels. Note that blanks arrive as `""`, not NULL, so `NULLIF` is needed or coverage looks like 100%.

In [2]:
cols = ["source_customer_id", "email_hashed", "customer_name_hashed", "phone_hashed",
        "ship_address_1_hashed", "ship_zip", "card_fingerprint"]
q("SELECT lower(channel) AS channel, " + ", ".join(f"round(100.0*count(NULLIF({c},''))/count(*),1) AS {c.replace('_hashed','')}" for c in cols)
  + " FROM raw GROUP BY 1").set_index("channel").T

channel,amazon,shopify
source_customer_id,0.0,100.0
email,1.0,100.0
customer_name,54.5,100.0
phone,0.0,65.9
ship_address_1,54.5,100.0
ship_zip,100.0,100.0
card_fingerprint,0.0,20.7


**Read:** the marketplace has no customer ID, phone or card, and almost no email. Its only bridge to the web store is **name + address + zip**, and a large share of its orders carry nothing but a zip. That share is the hard ceiling on what any matching can do.

### 1b. Is the email hash safe to match on?
Emails arrive as `sha256(local_part)@domain`. If only the local part is hashed, generic addresses like `info@` hash identically across unrelated businesses.

In [3]:
import hashlib
generic = {hashlib.sha256(w.encode()).hexdigest(): w for w in ["info", "hello", "contact", "sales"]}
top = q("""SELECT split_part(email_hashed,'@',1) AS local_hash, count(DISTINCT split_part(email_hashed,'@',2)) AS domains
             FROM raw WHERE email_hashed <> '' GROUP BY 1 ORDER BY 2 DESC LIMIT 5""")
top["decoded"] = top.local_hash.map(generic)
top

,local_hash,domains,decoded
0,2cf24dba5fb0a30e26e83b2ac5b9e29e1b161e5c1fa742...,49,hello
1,093e7d5fdbaacfa92448861b11075b3fba532a07d945b4...,45,contact
2,e04eb29020eaa961e99d3162635e9fe9585c5a1121bd88...,42,sales
3,06271baf49532c879aa3c58b48671884bcc858f0919741...,32,info
4,18ecdef7797bef82867d6381cf72f375f84e0b1e242244...,1,NaN


**Read:** the most-shared hashes decode to `info`, `hello` and so on, each spread across many domains. **Decision: match on the full `hash@domain` string, never the hash alone.**

### 1c. Is the name hash consistent?
A repeat customer should have the same name hash on every order. Checking this on each channel separately:

In [4]:
q("""
SELECT 'web store: repeat customer IDs' AS test, round(100.0*avg(CASE WHEN nn=1 THEN 1 ELSE 0 END),1) AS pct_consistent
FROM (SELECT source_customer_id, count(DISTINCT customer_name_hashed) nn FROM raw
      WHERE channel='shopify' AND source_customer_id<>'' GROUP BY 1 HAVING count(*)>=2)
UNION ALL
SELECT 'marketplace: repeat addresses', round(100.0*avg(CASE WHEN nn=1 THEN 1 ELSE 0 END),1)
FROM (SELECT ship_address_1_hashed, count(DISTINCT customer_name_hashed) nn FROM raw
      WHERE channel='amazon' AND customer_name_hashed<>'' GROUP BY 1 HAVING count(*)>=2)
""")

,test,pct_consistent
0,web store: repeat customer IDs,0.9
1,marketplace: repeat addresses,62.4


**Read:** web-store name hashes change on every order (the generator simulates a salted-hash defect, which does happen upstream), while marketplace names mostly repeat (the misses are two people at one address). **Decision: use name only inside the marketplace**, and flag the defect, because fixing it would turn many household-level matches into person-level ones.

### 1d. Hub values
A value shared by far more customers than any real person or household is junk: a store phone number, an apartment building, a reshipper. Left in, one such value chains thousands of strangers together.

In [5]:
q("""
SELECT 'phone' AS field, phone_hashed AS value, count(DISTINCT source_customer_id) AS customer_ids
FROM raw WHERE phone_hashed<>'' GROUP BY 2 ORDER BY 3 DESC LIMIT 3
""")

,field,value,customer_ids
0,phone,63348f9c667c95f29f202500c1d9fec197fc1f72033de4...,186
1,phone,f6a25babb1ab758f653ebfefeba521f5d092f321cec21e...,3
2,phone,caae2e67fb9908a08db535466b0093d052b8b0deab4fdc...,3


**Decision:** ignore any value shared by more than `MAX_ENTITIES` (5) entities.

### 1e. How trustworthy is each key?
For every group of orders sharing a key, how often do they also share an email? Email acts as the reference. This is a **lower bound** on precision, because one person can use two emails.

In [6]:
keys = {"card": "card_fingerprint", "phone": "phone_hashed",
        "address + zip": "ship_address_1_hashed || left(regexp_replace(ship_zip,'[^0-9]','','g'),5)"}
out = []
for label, expr in keys.items():
    out.append(q(f"""
      WITH t AS (SELECT NULLIF({expr},'') k, email_hashed e FROM raw WHERE email_hashed<>'' AND channel='shopify'),
           g AS (SELECT k, count(DISTINCT e) ne FROM t WHERE k IS NOT NULL GROUP BY k HAVING count(*)>=2)
      SELECT '{label}' AS key, count(*) AS groups, round(100.0*avg(CASE WHEN ne=1 THEN 1 ELSE 0 END),1) AS pct_single_email FROM g"""))
pd.concat(out)

,key,groups,pct_single_email
0,card,1263,94.8
0,phone,3890,93.3
0,address + zip,5847,92.9


**Read:** card and phone point to one person. Address + zip is less reliable, because families and roommates share an address, and on real data the gap is usually wider than here. **Decision: two ID levels.**
- `person_id`: customer ID, full email, card, payment reference, phone. Use it for KPIs.
- `household_id`: adds address + zip. Use it for suppression lists and household retention.

## 2. Resolve
The rules live in one config block. Matches are joined transitively with connected components: if A ~ B on email and B ~ C on card, then A, B and C are one person.

In [7]:
print("MAX_ENTITIES =", MAX_ENTITIES)
pd.DataFrame([(k, v[0]) for k, v in RULES.items()], columns=["rule", "level"])

MAX_ENTITIES = 5


,rule,level
0,email,person
1,card,person
2,par,person
3,phone,person
4,addr_zip,household


In [8]:
con2, m = run(DATA, "../output")
m

{'orders': 33924,
 'source_entities': 24396,
 'persons': 24066,
 'households': 21978,
 'pct_marketplace_unresolvable': 45.6,
 'largest_person_cluster': 4,
 'hub_values_blocked': {'phone': 1, 'addr_zip': 1},
 'passes': {'person': 3, 'household': 3},
 'runtime_sec': 0.9}

## 3. Validate: does it over-merge?

In [9]:
con2.sql("""
SELECT entities, count(*) AS persons FROM (SELECT person_id, count(DISTINCT entity) entities FROM identity GROUP BY 1)
GROUP BY 1 ORDER BY 1""").df()

,entities,persons
0,1,23752
1,2,299
2,3,14
3,4,1


**Read:** almost every person is one or two source entities, and the largest cluster is only a few entities. The hub guardrail prevents runaway chains.

## 4. What changes
The same orders under three definitions of "customer". **Native** = what reporting sees today: the web-store customer ID, with every marketplace order counted as its own unknown customer.

In [10]:
con2.sql("""CREATE OR REPLACE TABLE ident AS SELECT *,
  CASE WHEN entity LIKE 'S:%' THEN entity ELSE 'O:' || order_id END AS native_id FROM identity""")
rows = []
for name, col in {"native": "native_id", "person": "person_id", "household": "household_id"}.items():
    rows.append(con2.sql(f"""
      WITH c AS (SELECT {col} k, count(*) n, sum(order_value) rev FROM ident GROUP BY 1)
      SELECT '{name}' AS definition, count(*) AS customers,
             round(100.0*avg(CASE WHEN n>=2 THEN 1 ELSE 0 END),1) AS repeat_rate_pct,
             round(avg(rev),2) AS revenue_per_customer FROM c""").df())
pd.concat(rows, ignore_index=True)

,definition,customers,repeat_rate_pct,revenue_per_customer
0,native,24670,23.3,92.17
1,person,24066,25.5,94.48
2,household,21978,31.4,103.46


In [11]:
# "New" customers that weren't new: first order under the native ID, but not the person's first order
con2.sql("""
WITH f AS (SELECT *,
  row_number() OVER (PARTITION BY native_id ORDER BY processed_at, order_id) = 1 AS new_native,
  row_number() OVER (PARTITION BY person_id ORDER BY processed_at, order_id) = 1 AS new_person
  FROM ident)
SELECT channel, count(*) FILTER (WHERE new_native) AS counted_as_new,
       count(*) FILTER (WHERE new_native AND NOT new_person) AS actually_returning,
       round(100.0*count(*) FILTER (WHERE new_native AND NOT new_person)/count(*) FILTER (WHERE new_native),1) AS pct
FROM f GROUP BY 1""").df()

,channel,counted_as_new,actually_returning,pct
0,shopify,17801,305,1.7
1,amazon,6869,299,4.4


**Why it matters financially:**
- Every returning customer counted as "new" inflates the new-customer denominator, so **reported CAC is understated** by the same ratio.
- Those customers can be **suppressed from prospecting audiences**, which frees up to `returning × CAC` in acquisition spend.
- Marketplace repeat buying, invisible today, becomes measurable.

## 5. Running it in production
The same SQL runs incrementally on each new file drop:
1. Load only new files and upsert on `(channel, order_id)`.
2. Relabel only the components the new values touch.
3. When two customers merge, keep the older ID and record the merge history.
4. Monitor the merge rate, the largest cluster and the hub counts from `run_summary.json`.